# Mixray — Colab quality lab

A standalone offline experiment notebook. No Modal, Meta hosted API, public repository, or production deployment is required.
Upload this notebook through **Colab → File → Upload notebook**. Run cells individually, not Run all.

**Start on CPU** for downloads/preparation. Select **A100 GPU** for the SAM stage, then return to CPU for Demucs/Basic Pitch if convenient. Changing runtime can reset `/content`; Drive files survive, so rerun setup cells after reconnecting. The notebook creates isolated Python 3.11 projects instead of changing Colab's NumPy/TensorFlow environment.

First run: **one development clip, two sequential candidates**. Do not spend the entire 220-unit balance on a sweep. Read the actual units/hour in Colab's resource panel; allocation and consumption are variable. This notebook limits subprocess time, not billing units. Disconnect when finished.

Scope: real-stem mixture construction; SAM large or large-tv candidate WAVs; Judge on the same pool; Demucs fallback; Basic Pitch activations/raw notes; aligned SI-SDR/SI-SDR improvement and note/pitch-class F1 when independent note labels exist. **This is a diagnostic baseline, not the production arrangement pipeline or the complete Q0–Q5 plan.** It does not fix application code. AlbumDB provides stems, not assumed note annotations. Missing note labels remain null.

Validated locally: code compilation, offline audio/metric/cache controls, and Linux dependency resolution for Basic Pitch and Demucs. SAM dependency resolution did not complete because the local disk filled. **Colab dependency resolution and GPU inference still require the first smoke run.** Setup fails visibly rather than silently downgrading a model.

References: [Colab FAQ](https://research.google.com/colaboratory/faq.html), [SAM Audio](https://github.com/facebookresearch/sam-audio), [AlbumDB](https://research.aalto.fi/en/datasets/albumdb-a-multitrack-dataset-of-a-ten-song-album-with-stereo-and-/), [Demucs](https://github.com/facebookresearch/demucs), [Basic Pitch](https://github.com/spotify/basic-pitch).

## 1. Persistent workspace and small initial budget
Grant Drive access. All measured audio and reports go to your own Drive. Never publish the notebook with tokens or private audio outputs. Keep existing downloads; downloading from Colab may help network routing but cannot guarantee faster Zenodo service.

In [ ]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import logging
import os
import re
import shutil
import subprocess
import sys
import time
import urllib.request
import zipfile
from google.colab import drive

import logging
import sys


def get_logger(name: str = "mixray", level: int = logging.INFO) -> logging.Logger:
    log = logging.getLogger(name)
    if not log.handlers:  # re-running the cell must not add a second handler
        handler = logging.StreamHandler(sys.stdout)
        handler.setFormatter(
            logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S")
        )
        log.addHandler(handler)
    log.setLevel(level)
    log.propagate = False  # keep Colab's root handler from printing each line twice
    return log


log = get_logger()

drive.mount("/content/drive")
BASE = Path("/content/drive/MyDrive/mixray-quality")
DATA = BASE / "datasets" / "albumdb"
WORK = Path("/content/mixray-quality-work")
RUN_ID = "smoke-001"  # New ID when code, input, normalization or environment changes.
MODEL_ID = "facebook/sam-audio-large"  # Change only in a separate experiment.
DRAWS = 2
MAX_CASES = 1
MAX_STAGE_SECONDS = 900
UNITS_BEFORE = 220.0  # Replace with the actual resource-panel balance.
RUN = BASE / "runs" / RUN_ID
for directory in (BASE, DATA, WORK, RUN):
    directory.mkdir(parents=True, exist_ok=True)
log.info("Reports: %s", RUN)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
22:37:17 | INFO    | Reports: /content/drive/MyDrive/mixray-quality/runs/smoke-001


## 2. CPU dependency and native-library preflight
Only `uv` is bootstrapped with pip into an isolated tool directory. Model packages are declared in generated, exactly pinned `pyproject.toml` files and installed through `uv sync`. Lockfiles are preserved with the run.

The SAM Git revisions and compatibility pins match `engine/images.py`. NumPy 1.26.4, Transformers 4.57.1, Hugging Face Hub 0.36.2, protobuf 3.20.3 and setuptools 80.9.0 are retained for the documented upstream constraints. TensorFlow 2.15.0 is required by Basic Pitch's Python 3.11 peer range. Torch/vision/audio mirror Colab's preinstalled matched wheels to preserve its CUDA-driver compatibility. A CUDA allocation change requires another smoke test.

`uv` is a setup tool; `mir-eval` is an offline scoring dependency already used by this repository. No runtime service dependencies are added. The SAM dependency overrides reproduce the existing protobuf repair and pin upstream's otherwise floating Git dependencies; smoke-test imports before spending on inference.

Run this **before downloading AlbumDB or choosing clips**, on a CPU runtime. It resolves all three dependency graphs, installs the environments, imports each model library and exercises TorchCodec's FFmpeg decoding with a tiny generated WAV. Only then should you allocate GPU time. Setup requires at least 20 GiB local free space. The final GPU cell separately tests a CUDA matrix multiplication before loading model weights.

Verification status (2026-10-31): the exact Basic Pitch and Demucs graphs resolved locally for Linux x86_64 / Python 3.11. SAM resolution was interrupted by local disk exhaustion; it has **not** passed. Linux installation/import and Colab CUDA/model inference have **not** been executed by the author. Docker was unavailable locally. Do not treat the offline metric tests as installation certification.

The first runtime's Torch versions are saved in `torch-pins.json` and reused after a CPU→GPU switch. A `+cpu` suffix is removed so a later GPU session can use that version's default CUDA wheel. The CUDA smoke test must still pass; this is not a guarantee about the new runtime's driver. If compatibility fails, preserve the error and make a separately versioned environment rather than changing pins silently.


In [ ]:
# Check prerequisites before any environment or model download.
for binary in ("git", "ffmpeg"):
    if shutil.which(binary) is None:
        raise RuntimeError(f"Missing {binary}; install the system package before continuing")
    subprocess.run([binary, "-version" if binary == "ffmpeg" else "--version"],
                   check=True, capture_output=True, timeout=20)
if shutil.disk_usage(WORK).free < 20 * 2**30:
    raise RuntimeError("Need at least 20 GiB free local disk for the isolated model environments")

def latest_version(package: str) -> str:
    with urllib.request.urlopen(f"https://pypi.org/pypi/{package}/json", timeout=30) as response:
        version = json.load(response)["info"]["version"]
    if not re.fullmatch(r"\d+(?:\.\d+)+", version):
        raise ValueError("Expected a stable numeric PyPI version")
    return version

bootstrap = RUN / "bootstrap.json"
if bootstrap.exists():
    uv_version = json.loads(bootstrap.read_text())["uv"]
else:
    uv_version = latest_version("uv")
    bootstrap.write_text(json.dumps({"uv": uv_version}))
uv_target = WORK / "uv-tool"
UV = str(uv_target / "bin" / "uv")
if not Path(UV).exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                    "--no-deps", "--target", str(uv_target), f"uv=={uv_version}"], check=True)
subprocess.run([UV, "--version"], check=True)

common = ["numpy==1.26.4", "soundfile==0.14.0", "librosa==0.11.0", "mir-eval==0.8.2"]
torch_manifest = RUN / "torch-pins.json"
if torch_manifest.exists():
    torch_pins = json.loads(torch_manifest.read_text())
else:
    torch_pins = {}
    for name in ("torch", "torchvision", "torchaudio"):
        version = importlib.metadata.version(name)
        # A CPU Colab kernel must not force a CPU-only SAM wheel into the later GPU runtime.
        torch_pins[name] = version.removesuffix("+cpu")
    torch_manifest.write_text(json.dumps(torch_pins, indent=2))
git_deps = [
    "sam-audio @ git+https://github.com/facebookresearch/sam-audio.git@bb4c6999d2677c7402360e426afc01ddfad6dce0",
    "dacvae @ git+https://github.com/facebookresearch/dacvae.git@414c20785fc3a28373073ea8ef7a1316eeeaca6e",
    "imagebind @ git+https://github.com/facebookresearch/ImageBind.git@53680b02d7e37b19b124fa37bae4b6c98c38f5be",
    "laion-clap @ git+https://github.com/lematt1991/CLAP.git@b85cb7ecac0af7d064cbf03ebd990e709e959fe7",
    "perception-models @ git+https://github.com/facebookresearch/perception_models@e72b6810b1133e1c879f2cc965d276eb73803f1f",
]
sam_compat = ["transformers==4.57.1", "huggingface-hub==0.36.2", "protobuf==3.20.3", "tensorboard==2.20.0", "setuptools==80.9.0"]
projects = {
    "pitch": common + ["basic-pitch==0.4.0", "tensorflow==2.15.0", "setuptools==80.9.0"],
    "demucs": common + ["demucs==4.1.0", "torch==2.15.0"],
    "sam": common + git_deps + sam_compat + [f"{name}=={version}" for name, version in torch_pins.items()],
}
PROJECTS = {}
RELOCK_PROJECTS = set()
for name, deps in projects.items():
    project = WORK / name
    project.mkdir(exist_ok=True)
    text = '[project]\nname = "mixray-quality-' + name + '"\nversion = "0.1.0"\nrequires-python = "==3.11.*"\ndependencies = ' + json.dumps(deps) + '\n[tool.uv]\npackage = false\n' + 'environments = ' + json.dumps(["sys_platform == 'linux' and platform_machine == 'x86_64'"]) + '\n'
    if name == "sam":
        text += "override-dependencies = " + json.dumps(git_deps[1:] + ["protobuf==3.20.3"]) + "\n"
        for package, version in torch_pins.items():
            if "+cu" in version:
                channel = version.split("+", 1)[1]
                text += f'\n[[tool.uv.index]]\nname = "{package}-cuda"\nurl = "https://download.pytorch.org/whl/{channel}"\nexplicit = true\n'
        text += "\n[tool.uv.sources]\n"
        for package, version in torch_pins.items():
            if "+cu" in version:
                text += f'{package} = {{ index = "{package}-cuda" }}\n'
    if name == "demucs":
        text += '\n[[tool.uv.index]]\nname = "cpu"\nurl = "https://download.pytorch.org/whl/cpu"\nexplicit = true\n[tool.uv.sources]\ntorch = { index = "cpu" }\n'
    saved = RUN / (name + "-pyproject.toml")
    if saved.exists() and saved.read_text() != text:
        previous = saved.read_text()
        tensorboard_only = name == "sam" and text.replace('"tensorboard==2.20.0", ', "", 1) == previous
        has_candidates = any(RUN.glob("*/sam-audio-*/candidate-*.wav"))
        if not tensorboard_only or has_candidates:
            raise ValueError("Runtime package pins changed; start a new RUN_ID")
        shutil.copy2(saved, RUN / "sam-pyproject.before-tensorboard-fix.toml")
        previous_lock = RUN / "sam-uv.lock"
        if previous_lock.exists():
            shutil.copy2(previous_lock, RUN / "sam-uv.before-tensorboard-fix.lock")
        RELOCK_PROJECTS.add(name)
        log.warning("Repairing the pre-inference TensorBoard/protobuf pin; previous configuration saved")
    saved.write_text(text)
    (project / "pyproject.toml").write_text(text)
    lock = RUN / (name + "-uv.lock")
    if lock.exists():
        shutil.copy2(lock, project / "uv.lock")
    PROJECTS[name] = project

ENV = dict(os.environ)
ENV.update({"HF_HOME": str(BASE / "weights"), "USE_TF": "0", "USE_FLAX": "0",
            "MPLBACKEND": "Agg", "TOKENIZERS_PARALLELISM": "false"})

def install_project(name: str) -> None:
    project = PROJECTS[name]
    command = [UV, "sync", "--project", str(project), "--python", "3.11"]
    if (project / "uv.lock").exists():
        command.append("--locked")
    subprocess.run(command, check=True, env=ENV, timeout=1800)
    shutil.copy2(project / "uv.lock", RUN / (name + "-uv.lock"))
    smoke_env = dict(ENV)
    smoke_env["CUDA_VISIBLE_DEVICES"] = "-1"
    if name == "pitch":
        smoke_env.pop("USE_TF", None)
    probes = {
        "pitch": "import tensorflow; from basic_pitch.inference import predict; import librosa, soundfile, mir_eval",
        "demucs": "import torch; import demucs.separate; from demucs.pretrained import get_model",
        "sam": "from tensorboard.compat.proto import event_pb2; import torch, torchaudio, torchvision; from sam_audio import SAMAudio, SAMAudioProcessor; "
               "from sam_audio.ranking import create_ranker; from torchcodec.decoders import AudioDecoder; "
               "import tempfile, soundfile, numpy; "
               "f = tempfile.NamedTemporaryFile(suffix='.wav'); "
               "soundfile.write(f.name, numpy.zeros(4800), 48000); "
               "a = AudioDecoder(f.name).get_all_samples().data; assert a.shape[-1] == 4800",
    }
    probe = "import sys; assert sys.version_info[:2] == (3, 11); " + probes[name]
    result = subprocess.run(
        [UV, "run", "--no-sync", "--project", str(project), "python", "-c", probe],
        env=smoke_env, check=False, capture_output=True, text=True, timeout=180,
    )
    diagnostic = result.stdout + "\n" + result.stderr
    # Never store environment variables; only the import subprocess output is captured.
    token = smoke_env.get("HF_TOKEN", "")
    if token:
        diagnostic = diagnostic.replace(token, "[REDACTED]")
    (RUN / (name + "-import-check.log")).write_text(diagnostic, encoding="utf-8")
    if result.returncode:
        raise RuntimeError(
            f"{name} import/native-library check failed (exit {result.returncode}). "
            f"Full log: {RUN / (name + '-import-check.log')}\n" + diagnostic[-12000:]
        )
    (RUN / (name + "-import-check.json")).write_text(json.dumps({
        "status": "passed", "pyproject_sha256": hashlib.sha256((project / "pyproject.toml").read_bytes()).hexdigest(),
        "uv_lock_sha256": hashlib.sha256((project / "uv.lock").read_bytes()).hexdigest(),
    }, indent=2))
    log.info("%s installation and CPU import/native-library check passed", name)

# Resolve all three graphs before installing any model environment. No GPU or HF key needed.
for name, project in PROJECTS.items():
    command = [UV, "lock", "--project", str(project), "--python", "3.11"]
    if (project / "uv.lock").exists() and name not in RELOCK_PROJECTS:
        command.append("--locked")
    subprocess.run(command, env=ENV, check=True, timeout=900)
    shutil.copy2(project / "uv.lock", RUN / (name + "-uv.lock"))
    log.info("%s dependency graph resolved", name)
for name in ("pitch", "demucs", "sam"):
    install_project(name)
log.info("CPU PREFLIGHT PASSED. Dataset preparation and the GPU smoke test can follow.")

## 3. Optional direct Zenodo download — CPU only
The first cell lists the actual files and checksums instead of guessing an archive name. Set `ZENODO_FILE` to the **Mixed stems** archive key in the second cell. If already downloaded/extracted on Drive, skip both download and extraction. Keep the record's README/license with your data.

In [ ]:
ZENODO_RECORD = "19683001"
request = urllib.request.Request(f"https://zenodo.org/api/records/{ZENODO_RECORD}",
                                 headers={"User-Agent": "Mixray-quality-evaluation/1.0"})
with urllib.request.urlopen(request, timeout=60) as response:
    zenodo = json.load(response)
zenodo_files = {item["key"]: item for item in zenodo["files"]}
for name, item in zenodo_files.items():
    log.info("%s | %.2f GiB | %s", name, item["size"] / 2**30, item["checksum"])
(DATA / "zenodo-record.json").write_text(json.dumps(zenodo, indent=2))
print(zenodo_files)

{'stems_mixed.zip': {'id': 'c60f265f-fe89-4a06-8bf0-86cafa4b29d9', 'key': 'stems_mixed.zip', 'size': 4959457891, 'checksum': 'md5:a6ddc38004047879622542f41ab691ed', 'links': {'self': 'https://zenodo.org/api/records/19683001/files/stems_mixed.zip/content'}}, 'masters_7.1.4.zip': {'id': 'be2741c0-f26e-4e4f-b6a1-d2d7497df3e3', 'key': 'masters_7.1.4.zip', 'size': 1750668978, 'checksum': 'md5:d9498bfaba5388b450133537df42b34d', 'links': {'self': 'https://zenodo.org/api/records/19683001/files/masters_7.1.4.zip/content'}}, 'stems_mixed_dry.zip': {'id': '8e7b37ab-e785-4319-bfdf-6b2727b7cb95', 'key': 'stems_mixed_dry.zip', 'size': 4596886118, 'checksum': 'md5:5857b6d38a69cad42131a96416e25bcc', 'links': {'self': 'https://zenodo.org/api/records/19683001/files/stems_mixed_dry.zip/content'}}, 'example_usage_automatic_mixing.zip': {'id': '4fd2609e-f7c3-43c0-94d4-857650df126a', 'key': 'example_usage_automatic_mixing.zip', 'size': 44655222, 'checksum': 'md5:f1b3d1195d94bd2aa1c42c559a0e3abd', 'links': {

In [ ]:
ZENODO_FILE = "stems_mixed.zip"  # Paste the exact Mixed archive key shown above, not the record URL.
if ZENODO_FILE:
    if ZENODO_FILE not in zenodo_files or Path(ZENODO_FILE).name != ZENODO_FILE:
        raise ValueError("Select an exact file key from the record")
    item = zenodo_files[ZENODO_FILE]
    archive = DATA / ZENODO_FILE
    subprocess.run(["curl", "-fL", "-C", "-", "--retry", "5", "--retry-delay", "15",
                    "--max-time", "3600", "-o", str(archive), item["links"]["self"]],
                   check=True, timeout=3700)
    algorithm, expected = item["checksum"].split(":", 1)
    with archive.open("rb") as handle:
        actual = hashlib.file_digest(handle, algorithm).hexdigest()
    if actual != expected:
        raise RuntimeError("Archive checksum mismatch; do not extract")
    log.info("Verified archive: %s", archive)
else:
    log.info("Download skipped. Set ZENODO_FILE or copy your existing archive into DATA.")

In [ ]:
ARCHIVE_NAME = "stems_mixed.zip"  # Exact ZIP filename inside DATA. Leave blank if already extracted.
if ARCHIVE_NAME:
    archive = DATA / ARCHIVE_NAME
    if archive.parent.resolve() != DATA.resolve():
        raise ValueError("Archive must be inside DATA")
    with zipfile.ZipFile(archive) as bundle:
        members = bundle.infolist()
        total = sum(member.file_size for member in members)
        if total > 30 * 2**30:
            raise ValueError("Archive exceeds the 30 GiB extraction cap; inspect before proceeding")
        for member in members:
            destination = (DATA / member.filename).resolve()
            if not destination.is_relative_to(DATA.resolve()):
                raise ValueError("Unsafe archive member path")
            if (member.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError("Archive symlinks are not accepted")
        bundle.extractall(DATA)
    log.info("Extracted %.2f GiB", total / 2**30)
wavs = sorted(DATA.rglob("*.wav")) + sorted(DATA.rglob("*.WAV"))
log.info("Found %d WAV files. First 80 paths:", len(wavs))
for path in wavs[:80]:
    log.info("%s", path.relative_to(DATA))

In [ ]:
log.info("%s", "hello")

22:37:24 | INFO    | hello


## 4. Explicit development cases
Choose synchronized **Mixed** stems, not a mastered stereo mix paired with raw dry stems. Include **all desired mix stems** in `stems`, including the target stems; `targets` is their exact subset. The worker sums them with a common gain, uses the same 30-second crop origin and preserves stereo for Demucs.

Example (replace every path, then assign `CASES = [...]`):
```python
{
  "id": "song01_guitar", "song": "albumdb_song01", "split": "dev",
  "instrument": "guitar", "prompt": "acoustic guitar",
  "source_url": "https://zenodo.org/records/19683001", "license": "CC-BY-4.0",
  "stems": [str(DATA / "actual/path/Guitar.wav"), str(DATA / "actual/path/Bass.wav"),
            str(DATA / "actual/path/Vocals.wav"), str(DATA / "actual/path/Drums.wav")],
  "targets": [str(DATA / "actual/path/Guitar.wav")],
  "start": 30.0, "duration": 30.0, "notes": None
}
```
`instrument` uses `guitar`, `piano`, `bass`, `vocals`, `drums`, or `other` for the corresponding Demucs comparison. For a specific singer/guitarist, a whole-category Demucs stem is **not** an identity-matched reference; document that limitation. Independent note annotations, if available, are `[start_seconds, end_seconds, MIDI_pitch]` relative to the clip. Do not manufacture them from model predictions. Keep songs/performers grouped across dev/holdout. Holdout inference is disabled by default.

In [ ]:
# Does SAM Audio do anything Demucs cannot: tell two guitars apart inside one mix?
RUN_ID = "dev-003"
DRAWS = 2
MAX_CASES = 4
MAX_STAGE_SECONDS = 1800
RUN = BASE / "runs" / RUN_ID
RUN.mkdir(parents=True, exist_ok=True)

SONGS = DATA / "stems_mixed"


def stem_list(song: str, *names: str) -> list[str]:
    return [str(SONGS / song / name) for name in names]


SEAS = "07 Take On The Seas"
SEAS_MIX = stem_list(SEAS, "bass.wav", "drums_kick.wav", "drums_overheads.wav",
                     "vox_lead.wav", "guitar_acoustic_main.wav", "guitar_electric_main.wav")
LIGHT = "06 Let The Light In"
LIGHT_MIX = stem_list(LIGHT, "bass.wav", "drums_kick.wav", "drums_overheads.wav",
                      "vox_lead.wav", "guitar_electric_solo_c.wav",
                      "guitar_electric_rhythm_l.wav")

# Each pair shares one mixture and one offset, so only the prompt differs between the two cases.
DRAFT = [
    {"group": "seas_two_guitars", "id": "seas_guitar_acoustic", "song": SEAS,
     "instrument": "guitar", "prompt": "acoustic guitar", "stems": SEAS_MIX,
     "targets": stem_list(SEAS, "guitar_acoustic_main.wav")},
    {"group": "seas_two_guitars", "id": "seas_guitar_electric", "song": SEAS,
     "instrument": "guitar", "prompt": "electric guitar", "stems": SEAS_MIX,
     "targets": stem_list(SEAS, "guitar_electric_main.wav")},
    {"group": "light_two_guitars", "id": "light_guitar_solo", "song": LIGHT,
     "instrument": "guitar", "prompt": "lead electric guitar", "stems": LIGHT_MIX,
     "targets": stem_list(LIGHT, "guitar_electric_solo_c.wav")},
    {"group": "light_two_guitars", "id": "light_guitar_rhythm", "song": LIGHT,
     "instrument": "guitar", "prompt": "rhythm electric guitar", "stems": LIGHT_MIX,
     "targets": stem_list(LIGHT, "guitar_electric_rhythm_l.wav")},
]

missing = sorted({p for case in DRAFT for p in case["stems"] if not Path(p).is_file()})
if missing:
    raise FileNotFoundError("Fix these stem paths before spending GPU time:\n" + "\n".join(missing))

# dev-002 wasted a case because only the group's first target was checked for activity.
ENERGY_SCAN = r'''
import json, sys
import numpy as np
import soundfile as sf

WINDOW = 30
MIN_TARGET_TO_MIX_DB = -20.0
groups = json.loads(sys.stdin.read())
chosen = {}
for name, spec in groups.items():
    curves, rate, seconds = [], None, None
    for path in spec["targets"]:
        audio, sample_rate = sf.read(path, dtype="float64", always_2d=True)
        mono = audio.mean(axis=1)
        whole = len(mono) // sample_rate
        if whole < WINDOW:
            raise ValueError(name + ": target shorter than the clip")
        edges = np.arange(0, whole * sample_rate, sample_rate)
        per_second = np.add.reduceat(mono[: whole * sample_rate] ** 2, edges)
        cumulative = np.concatenate(([0.0], np.cumsum(per_second)))
        totals = cumulative[WINDOW:] - cumulative[:-WINDOW]
        if rate is None:
            rate, seconds = sample_rate, len(totals)
        elif sample_rate != rate:
            raise ValueError(name + ": targets disagree on sample rate")
        curves.append(totals[: min(seconds, len(totals))])
        seconds = min(seconds, len(totals))
    # A shared window must suit every target, so score each by its own best window.
    shares = np.stack([c[:seconds] / max(c[:seconds].max(), 1e-30) for c in curves])
    start = int(np.argmax(shares.min(axis=0)))
    begin, stop = start * rate, (start + WINDOW) * rate
    mixed = None
    for path in spec["stems"]:
        audio, _ = sf.read(path, dtype="float64", always_2d=True, start=begin, stop=stop)
        mono = audio.mean(axis=1)
        mixed = mono if mixed is None else mixed + mono
    mix_rms = float(np.sqrt(np.mean(mixed**2)))
    levels = {}
    for path, curve in zip(spec["targets"], curves):
        rms = float(np.sqrt(curve[start] / (WINDOW * rate)))
        relative = 20.0 * np.log10(max(rms, 1e-30) / max(mix_rms, 1e-30))
        levels[path] = {"rms": rms, "to_mix_db": float(relative)}
        if relative < MIN_TARGET_TO_MIX_DB:
            raise ValueError(
                f"{name}: {path} sits {relative:.1f} dB under the mixture at its own best "
                "window; choose a louder stem instead of measuring near-silence"
            )
    chosen[name] = {"start": float(start), "mix_rms": mix_rms, "levels": levels}
print("ENERGY_SCAN " + json.dumps(chosen))
'''

scan_request = {}
for case in DRAFT:
    group = scan_request.setdefault(case["group"], {"stems": case["stems"], "targets": []})
    group["targets"].extend(t for t in case["targets"] if t not in group["targets"])
scan_env = dict(ENV)
scan_env.pop("USE_TF", None)
scan_env["CUDA_VISIBLE_DEVICES"] = "-1"
completed = subprocess.run(
    [UV, "run", "--no-sync", "--project", str(PROJECTS["pitch"]), "python", "-c", ENERGY_SCAN],
    input=json.dumps(scan_request), env=scan_env,
    capture_output=True, text=True, check=True, timeout=1800,
)
marker = [line for line in completed.stdout.splitlines() if line.startswith("ENERGY_SCAN ")]
if len(marker) != 1:
    raise RuntimeError("Energy scan produced no offsets:\n" + completed.stdout + completed.stderr)
offsets = json.loads(marker[0].removeprefix("ENERGY_SCAN "))
(RUN / "offsets.json").write_text(json.dumps(offsets, indent=2))

CASES = []
for case in DRAFT:
    picked = offsets[case["group"]]
    level = picked["levels"][case["targets"][0]]
    log.info("%-22s start %5.0f s  target %+6.1f dB vs mixture  prompt '%s'",
             case["id"], picked["start"], level["to_mix_db"], case["prompt"])
    CASES.append({
        "id": case["id"], "song": case["song"], "split": "dev",
        "instrument": case["instrument"], "prompt": case["prompt"],
        "source_url": "https://zenodo.org/records/19683001", "license": "CC-BY-4.0",
        "stems": case["stems"], "targets": case["targets"],
        "start": picked["start"], "duration": 30.0, "notes": None,
    })

if not CASES:
    raise ValueError("Add one real development case before installing/running models")
if len(CASES) > MAX_CASES or not 1 <= DRAWS <= 8:
    raise ValueError("Initial cap exceeded; inspect the smoke run before expanding")
CONFIG = {"output": str(RUN), "model": MODEL_ID, "draws": DRAWS,
          "allow_holdout": False, "cases": CASES}
config_path = RUN / "config.json"
if config_path.exists() and json.loads(config_path.read_text()) != CONFIG:
    raise ValueError("Configuration changed. Choose a new RUN_ID to retain the old baseline")
config_path.write_text(json.dumps(CONFIG, indent=2))
log.info("Wrote %d cases to %s", len(CASES), config_path)


23:24:04 | INFO    | seas_guitar_acoustic   start    81 s  target   -8.1 dB vs mixture  prompt 'acoustic guitar'
23:24:04 | INFO    | seas_guitar_electric   start    81 s  target  -12.0 dB vs mixture  prompt 'electric guitar'
23:24:04 | INFO    | light_guitar_solo      start   126 s  target   -8.3 dB vs mixture  prompt 'lead electric guitar'
23:24:04 | INFO    | light_guitar_rhythm    start   126 s  target  -14.5 dB vs mixture  prompt 'rhythm electric guitar'
23:24:04 | INFO    | Wrote 4 cases to /content/drive/MyDrive/mixray-quality/runs/dev-003/config.json


## 5. Install the embedded diagnostic worker
This cell only writes Python source. The worker saves every candidate before ranking, validates rates and durations, uses common normalization for raw-note comparisons and refuses mismatched caches. Float WAVs preserve unnormalized separation output. Candidate scoring is mono for comparability; Demucs input remains stereo. There is no arbitrary delay correction or time warping in this baseline.

In [ ]:
WORKER_SOURCE = r'''"""Offline Mixray diagnostic worker. Never contacts the deployed engine."""

from __future__ import annotations

import argparse
import hashlib
import json
import logging
import math
import re
import subprocess
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import TypeAlias

import numpy as np
import soundfile as sf
from numpy.typing import NDArray

LOGGER = logging.getLogger("mixray-quality")
RATE = 48000
Samples: TypeAlias = NDArray[np.float64]
NoteEvent: TypeAlias = tuple[float, float, int]


@dataclass(frozen=True)
class Case:
    identifier: str
    prompt: str
    instrument: str
    song: str
    split: str
    source_url: str
    license: str
    stems: tuple[Path, ...]
    targets: tuple[Path, ...]
    start: float
    duration: float
    notes: tuple[NoteEvent, ...] | None


def mapping(value: object) -> dict[str, object]:
    if not isinstance(value, dict) or not all(isinstance(k, str) for k in value):
        raise ValueError("Expected a JSON object")
    return value


def string(value: object) -> str:
    if not isinstance(value, str) or not value.strip():
        raise ValueError("Expected a nonempty string")
    return value


def number(value: object) -> float:
    if (
        isinstance(value, bool)
        or not isinstance(value, (float, int))
        or not math.isfinite(value)
    ):
        raise ValueError("Expected a finite number")
    return float(value)


def paths(value: object) -> tuple[Path, ...]:
    if not isinstance(value, list) or not value:
        raise ValueError("Specify at least one stem")
    result = tuple(Path(string(p)).resolve() for p in value)
    if len(set(result)) != len(result) or not all(p.is_file() for p in result):
        raise ValueError("Missing or duplicate stem path")
    return result


def parse_case(value: object) -> Case:
    data = mapping(value)
    identifier = string(data.get("id"))
    if not re.fullmatch(r"[a-zA-Z0-9_-]+", identifier):
        raise ValueError("Case IDs must use letters, digits, underscores or hyphens")
    stems, targets = paths(data.get("stems")), paths(data.get("targets"))
    if not set(targets).issubset(stems):
        raise ValueError("Target stems must also be included in the mixture")
    start, duration = number(data.get("start")), number(data.get("duration"))
    if start < 0 or not 0 < duration <= 30:
        raise ValueError("Use a positive clip of at most 30 seconds")
    raw_notes = data.get("notes")
    notes: list[NoteEvent] | None = None
    if raw_notes is not None:
        if not isinstance(raw_notes, list):
            raise ValueError("notes must be null or [start, end, MIDI pitch] rows")
        notes = []
        for row in raw_notes:
            if not isinstance(row, list) or len(row) != 3:
                raise ValueError("Invalid note row")
            onset, end, pitch = (number(v) for v in row)
            if (
                not 0 <= onset < end <= duration
                or not 0 <= pitch <= 127
                or pitch != int(pitch)
            ):
                raise ValueError(
                    "Note labels must be clip-relative seconds and integer MIDI"
                )
            notes.append((onset, end, int(pitch)))
    split = string(data.get("split"))
    if split not in {"dev", "holdout"}:
        raise ValueError("split must be dev or holdout")
    return Case(
        identifier,
        string(data.get("prompt")),
        string(data.get("instrument")),
        string(data.get("song")),
        split,
        string(data.get("source_url")),
        string(data.get("license")),
        stems,
        targets,
        start,
        duration,
        None if notes is None else tuple(notes),
    )


def digest(path: Path) -> str:
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


def save_json(path: Path, value: object) -> None:
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")
    temporary.replace(path)


def read_audio(path: Path) -> Samples:
    import librosa

    audio, rate = sf.read(path, dtype="float64", always_2d=True)
    if not np.isfinite(audio).all():
        raise ValueError(f"Nonfinite audio: {path.name}")
    mono = audio.mean(axis=1)
    expected = round(len(mono) * RATE / rate)
    resampled = librosa.resample(mono, orig_sr=rate, target_sr=RATE)
    if not expected <= len(resampled) <= expected + 1:
        raise ValueError("Unexpected resampler output duration")
    return np.asarray(resampled[:expected], dtype=np.float64)


def crop(path: Path, start: float, duration: float) -> Samples:
    import librosa

    with sf.SoundFile(path) as handle:
        if handle.channels not in (1, 2):
            raise ValueError("Use mono/stereo stems, not immersive masters")
        handle.seek(round(start * handle.samplerate))
        audio = handle.read(
            round(duration * handle.samplerate), dtype="float64", always_2d=True
        )
        if len(audio) != round(duration * handle.samplerate):
            raise ValueError(f"Stem is shorter than selected clip: {path.name}")
        audio = librosa.resample(audio.T, orig_sr=handle.samplerate, target_sr=RATE).T
    if audio.shape[1] == 1:
        audio = np.repeat(audio, 2, axis=1)
    if not np.isfinite(audio).all():
        raise ValueError("Nonfinite source samples")
    return np.asarray(audio[: round(duration * RATE)], dtype=np.float64)


def prepare(case: Case, root: Path) -> Path:
    directory = root / case.identifier
    directory.mkdir(parents=True, exist_ok=True)
    provenance = {
        "id": case.identifier,
        "prompt": case.prompt,
        "instrument": case.instrument,
        "song": case.song,
        "split": case.split,
        "source_url": case.source_url,
        "license": case.license,
        "start": case.start,
        "duration": case.duration,
        "notes": case.notes,
        "stems": {str(p): digest(p) for p in case.stems},
        "targets": [str(p) for p in case.targets],
        "rate": RATE,
    }
    manifest = directory / "case.json"
    if manifest.exists():
        old = mapping(json.loads(manifest.read_text()))
        if old.get("provenance") != json.loads(json.dumps(provenance)):
            raise ValueError(
                "Case changed. Use a new RUN_ID; do not mix incompatible results"
            )
        for name in ("mix.wav", "reference.wav"):
            if mapping(old.get("hashes")).get(name) != digest(directory / name):
                raise ValueError("Prepared audio checksum changed")
        return directory
    mix = np.zeros((round(case.duration * RATE), 2), dtype=np.float64)
    target = np.zeros_like(mix)
    for path in case.stems:
        audio = crop(path, case.start, case.duration)
        mix += audio
        if path in case.targets:
            target += audio
    peak = max(float(np.max(np.abs(mix))), float(np.max(np.abs(target))))
    if peak == 0:
        raise ValueError("The entire selected clip is silent")
    gain = min(1.0, 0.891 / peak)
    sf.write(directory / "mix.wav", mix * gain, RATE, subtype="FLOAT")
    sf.write(directory / "reference.wav", target * gain, RATE, subtype="FLOAT")
    save_json(
        manifest,
        {
            "provenance": provenance,
            "common_gain": gain,
            "hashes": {n: digest(directory / n) for n in ("mix.wav", "reference.wav")},
        },
    )
    return directory


def si_sdr(reference: Samples, estimate: Samples) -> float | None:
    if len(reference) != len(estimate):
        raise ValueError("SI-SDR inputs must have equal duration after real resampling")
    target = reference - reference.mean()
    audio = estimate - estimate.mean()
    power = float(target @ target)
    if power < 1e-12:
        return None
    projection = target * float(audio @ target) / power
    error = audio - projection
    return float(
        10
        * np.log10(
            (float(projection @ projection) + 1e-12) / (float(error @ error) + 1e-12)
        )
    )


def notes_f1(
    truth: tuple[NoteEvent, ...], detected: tuple[NoteEvent, ...], *, chroma: bool
) -> float:
    import mir_eval

    def arrays(events: tuple[NoteEvent, ...]) -> tuple[Samples, Samples]:
        intervals = np.asarray(
            [(s, e) for s, e, _ in events], dtype=np.float64
        ).reshape(-1, 2)
        pitches = [60 + p % 12 if chroma else p for _, _, p in events]
        hz = 440.0 * np.power(2.0, (np.asarray(pitches) - 69) / 12)
        return intervals, np.asarray(hz, dtype=np.float64)

    ti, tp = arrays(truth)
    di, dp = arrays(detected)
    _, _, f1, _ = mir_eval.transcription.precision_recall_f1_overlap(
        ti, tp, di, dp, onset_tolerance=0.05, pitch_tolerance=50.0, offset_ratio=None
    )
    return float(f1)


def sam(case: Case, directory: Path, model_id: str, draws: int) -> None:
    import torch
    from huggingface_hub import scan_cache_dir, snapshot_download
    from sam_audio import SAMAudio, SAMAudioProcessor
    from sam_audio.model.config import JudgeRankerConfig
    from sam_audio.ranking import create_ranker

    if not torch.cuda.is_available():
        raise RuntimeError("A CUDA GPU is required for the SAM stage")
    memory_gib = torch.cuda.get_device_properties(0).total_memory / 2**30
    if memory_gib < 35:
        raise RuntimeError(
            "Select an A100 with at least 40 GB. No silent model downgrade"
        )
    artifact = directory / model_id.split("/")[-1]
    artifact.mkdir(exist_ok=True)
    revision_path = artifact / "model.json"
    revision = None
    if revision_path.exists():
        revision = string(
            mapping(json.loads(revision_path.read_text())).get("revision")
        )
    snapshot = snapshot_download(
        model_id, revision=revision, allow_patterns=["*.json", "*.pt", "*.safetensors"]
    )
    save_json(
        revision_path,
        {
            "model": model_id,
            "revision": Path(snapshot).name,
            "torch": torch.__version__,
            "gpu": torch.cuda.get_device_name(0),
            "memory_gib": memory_gib,
            "precision": "fp32",
        },
    )
    model = (
        SAMAudio.from_pretrained(
            snapshot, visual_ranker=None, text_ranker=None, span_predictor=None
        )
        .eval()
        .cuda()
    )
    processor = SAMAudioProcessor.from_pretrained(snapshot)
    mix = read_audio(directory / "mix.wav")
    batch = processor(
        audios=[torch.tensor(mix, dtype=torch.float32).unsqueeze(0)],
        descriptions=[case.prompt],
    ).to("cuda")
    for index in range(draws):
        target_file = artifact / f"candidate-{index}.wav"
        receipt = artifact / f"candidate-{index}.json"
        if receipt.exists():
            stored = mapping(json.loads(receipt.read_text()))
            if stored.get("sha256") != digest(target_file):
                raise ValueError("Candidate cache checksum mismatch")
            continue
        seed = 1000 + index
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
        started = time.monotonic()
        with torch.inference_mode():
            result = model.separate(batch, predict_spans=False, reranking_candidates=1)
        torch.cuda.synchronize()
        seconds = time.monotonic() - started
        target = result.target[0].detach().float().cpu().numpy().reshape(-1)
        residual = result.residual[0].detach().float().cpu().numpy().reshape(-1)
        if len(target) < len(mix) or len(residual) < len(mix):
            raise ValueError(
                "Model shortened the clip; inspect alignment instead of padding silently"
            )
        if not np.isfinite(target).all() or not np.isfinite(residual).all():
            raise ValueError("Model returned nonfinite audio")
        sf.write(target_file, target[: len(mix)], RATE, subtype="FLOAT")
        sf.write(
            artifact / f"residual-{index}.wav",
            residual[: len(mix)],
            RATE,
            subtype="FLOAT",
        )
        save_json(
            receipt,
            {
                "seed": seed,
                "seconds": seconds,
                "sha256": digest(target_file),
                "peak_vram_gib": torch.cuda.max_memory_allocated() / 2**30,
            },
        )
        LOGGER.info("Saved %s candidate %d in %.1fs", case.identifier, index, seconds)
    # Score saved waveforms in a separate memory phase, one candidate at a time.
    del model, processor, batch
    import gc

    gc.collect()
    torch.cuda.empty_cache()
    judge = create_ranker(JudgeRankerConfig())
    if judge is None:
        raise RuntimeError("Judge ranker unavailable")
    judge = judge.eval().cuda()
    scores: dict[str, float] = {}
    axes_by_candidate: dict[str, dict[str, float]] = {}
    tensor_mix = torch.tensor(mix, dtype=torch.float32, device="cuda").unsqueeze(0)
    for name, path in [("mix", directory / "mix.wav")] + [
        (f"candidate-{i}", artifact / f"candidate-{i}.wav") for i in range(draws)
    ]:
        tensor = torch.tensor(
            read_audio(path), dtype=torch.float32, device="cuda"
        ).unsqueeze(0)
        with torch.inference_mode():
            processed = judge.processor(
                text=[case.prompt],
                input_audio=[tensor_mix],
                separated_audio=[tensor],
                return_tensors="pt",
                padding=True,
                sampling_rate=RATE,
            )
            judged = judge.model(**processed.to("cuda"))
        axes = {
            axis: float(getattr(judged, axis).reshape(-1)[0].cpu())
            for axis in ("overall", "precision", "recall", "faithfulness")
        }
        if not all(math.isfinite(value) for value in axes.values()):
            raise ValueError("Judge returned nonfinite axes")
        axes_by_candidate[name] = axes
        value = axes["overall"]
        if not math.isfinite(value):
            raise ValueError("Judge returned a nonfinite score")
        scores[name] = value
    save_json(
        artifact / "huggingface-snapshots.json",
        {
            repo.repo_id: sorted(revision.commit_hash for revision in repo.revisions)
            for repo in scan_cache_dir().repos
        },
    )
    save_json(
        artifact / "judge.json",
        {
            "scores": scores,
            "axes": axes_by_candidate,
            "selected": max(scores, key=lambda key: scores[key]),
            "calibrated_probability": False,
        },
    )


def demucs(directory: Path) -> None:
    output = directory / "demucs"
    marker = output / "done.json"
    if marker.exists():
        for name, checksum in mapping(json.loads(marker.read_text())).items():
            if digest(output / name) != checksum:
                raise ValueError("Demucs cache checksum mismatch")
        return
    subprocess.run(
        [
            sys.executable,
            "-m",
            "demucs.separate",
            "-n",
            "htdemucs_6s",
            "--device",
            "cpu",
            "--float32",
            "--shifts",
            "0",
            "--out",
            str(output),
            str(directory / "mix.wav"),
        ],
        check=True,
        timeout=900,
    )
    outputs = list(output.rglob("*.wav"))
    if len(outputs) != 6:
        raise ValueError("Expected six Demucs stems")
    save_json(marker, {str(p.relative_to(output)): digest(p) for p in outputs})


def waveforms(case: Case, directory: Path, model_id: str) -> dict[str, Path]:
    paths_by_name = {
        "mix": directory / "mix.wav",
        "reference": directory / "reference.wav",
    }
    artifact = directory / model_id.split("/")[-1]
    for path in sorted(artifact.glob("candidate-*.wav")):
        paths_by_name[path.stem] = path
    fallback = directory / "demucs" / "htdemucs_6s" / "mix" / f"{case.instrument}.wav"
    if fallback.exists():
        paths_by_name["demucs"] = fallback
    return paths_by_name


def transcribe(case: Case, directory: Path, model_id: str) -> None:
    from basic_pitch.inference import predict

    output = directory / model_id.split("/")[-1] / "transcription"
    output.mkdir(parents=True, exist_ok=True)
    for name, path in waveforms(case, directory, model_id).items():
        destination = output / f"{name}.json"
        source_hash = digest(path)
        if destination.exists():
            if (
                mapping(json.loads(destination.read_text())).get("audio_sha256")
                != source_hash
            ):
                raise ValueError("Transcription source changed; use a new run")
            continue
        audio = read_audio(path)
        peak = float(np.max(np.abs(audio)))
        gain = 0.891 / peak if peak > 0.01 else 1.0
        normalized = output / f"{name}-normalized.wav"
        sf.write(normalized, audio * gain, RATE, subtype="PCM_16")
        activation, _, events = predict(
            str(normalized),
            onset_threshold=0.5,
            frame_threshold=0.3,
            minimum_note_length=30.0,
            melodia_trick=True,
        )
        np.savez_compressed(output / f"{name}-activations.npz", **activation)
        notes = [(float(s), float(e), int(p)) for s, e, p, _, _ in events]
        save_json(
            destination,
            {
                "audio_sha256": source_hash,
                "gain": gain,
                "notes": notes,
                "decoder": "Basic Pitch 0.4.0; onset .5; frame .3; min 30ms; melodia on",
                "scope": "raw notes; no production stem filters or arrangement",
            },
        )


def report(case: Case, directory: Path, model_id: str) -> None:
    reference = read_audio(directory / "reference.wav")
    baseline = si_sdr(reference, read_audio(directory / "mix.wav"))
    rows: list[dict[str, object]] = []
    artifact = directory / model_id.split("/")[-1]
    for name, path in waveforms(case, directory, model_id).items():
        score = si_sdr(reference, read_audio(path))
        row: dict[str, object] = {
            "case": case.identifier,
            "song": case.song,
            "split": case.split,
            "instrument": case.instrument,
            "candidate": name,
            "si_sdr_db": score,
            "si_sdri_db": None
            if score is None or baseline is None
            else score - baseline,
        }
        note_path = artifact / "transcription" / f"{name}.json"
        row["note_f1"] = None
        row["pitch_class_f1"] = None
        if case.notes is not None and note_path.exists():
            raw = mapping(json.loads(note_path.read_text())).get("notes")
            if not isinstance(raw, list):
                raise ValueError("Invalid stored notes")
            detected: list[NoteEvent] = []
            for event in raw:
                if not isinstance(event, list) or len(event) != 3:
                    raise ValueError("Invalid stored note event")
                detected.append(
                    (number(event[0]), number(event[1]), int(number(event[2])))
                )
            row["note_f1"] = notes_f1(case.notes, tuple(detected), chroma=False)
            row["pitch_class_f1"] = notes_f1(case.notes, tuple(detected), chroma=True)
        rows.append(row)
    artifact.mkdir(exist_ok=True)
    save_json(artifact / "report.json", rows)
    import csv

    with (artifact / "report.csv").open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    LOGGER.info(
        "Saved %s; missing note labels are null, never invented",
        artifact / "report.json",
    )


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "stage", choices=["prepare", "sam", "demucs", "transcribe", "report"]
    )
    parser.add_argument("config", type=Path)
    args = parser.parse_args()
    config = mapping(json.loads(args.config.read_text()))
    root = Path(string(config.get("output")))
    model = string(config.get("model"))
    if model not in {"facebook/sam-audio-large", "facebook/sam-audio-large-tv"}:
        raise ValueError("Use a declared large checkpoint")
    draws_value = number(config.get("draws"))
    if draws_value != int(draws_value) or not 1 <= draws_value <= 8:
        raise ValueError("draws must be an integer in 1..8")
    raw_cases = config.get("cases")
    if not isinstance(raw_cases, list) or not 1 <= len(raw_cases) <= 24:
        raise ValueError("Configure between one and 24 cases")
    cases = [parse_case(value) for value in raw_cases]
    if len({case.identifier for case in cases}) != len(cases):
        raise ValueError("Duplicate case IDs")
    by_song: dict[str, str] = {}
    for case in cases:
        if case.song in by_song and by_song[case.song] != case.split:
            raise ValueError("The same song cannot occur in dev and holdout")
        by_song[case.song] = case.split
    if args.stage != "prepare" and not config.get("allow_holdout", False):
        cases = [case for case in cases if case.split == "dev"]
    if not cases:
        raise ValueError(
            "No development cases selected; holdout inference remains disabled"
        )
    started = time.monotonic()
    for case in cases:
        directory = root / case.identifier
        if args.stage == "prepare":
            prepare(case, root)
        elif not (directory / "case.json").is_file():
            raise ValueError("Prepare inputs before running inference")
        elif args.stage == "sam":
            sam(case, directory, model, int(draws_value))
        elif args.stage == "demucs":
            demucs(directory)
        elif args.stage == "transcribe":
            transcribe(case, directory, model)
        else:
            report(case, directory, model)
    LOGGER.info("%s finished in %.1fs", args.stage, time.monotonic() - started)


if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
    try:
        main()
    except Exception:
        LOGGER.exception("Stage failed; completed candidate files remain available")
        raise SystemExit(1) from None
'''
worker_path = WORK / "quality_worker.py"
worker_path.write_text(WORKER_SOURCE)
saved_worker = RUN / "quality_worker.py"
if saved_worker.exists() and saved_worker.read_text() != WORKER_SOURCE:
    raise ValueError("Worker changed; use a new RUN_ID")
saved_worker.write_text(WORKER_SOURCE)

def run_stage(stage: str, project_name: str) -> None:
    env = dict(ENV)
    if project_name == "pitch":
        env.pop("USE_TF", None)
        env["CUDA_VISIBLE_DEVICES"] = "-1"
    started = time.monotonic()
    command = [UV, "run", "--no-sync", "--project", str(PROJECTS[project_name]),
               "python", str(worker_path), stage, str(config_path)]
    status = "failed"
    try:
        import signal
        process = subprocess.Popen(command, env=env, start_new_session=True)
        try:
            result = process.wait(timeout=MAX_STAGE_SECONDS)
        except (subprocess.TimeoutExpired, KeyboardInterrupt):
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGKILL)
            process.wait()
            raise
        if result != 0:
            raise subprocess.CalledProcessError(result, command)
        status = "completed"
    finally:
        with (RUN / "stage-times.jsonl").open("a") as handle:
            handle.write(json.dumps({"stage": stage, "status": status,
                                     "seconds": time.monotonic() - started}) + "\n")

run_stage("prepare", "pitch")

## 6. SAM installation and access — select A100 before inference
Hugging Face access to `facebook/sam-audio-large` (or `-large-tv`) and `facebook/sam-audio-judge` must already be granted. Add **HF_TOKEN** in Colab's Secrets panel and grant notebook access. The token is passed only through the child process environment; do not paste it into a cell. There is no Modal key.

The GPU stage requires at least 35 GiB reported device memory and never falls back to small. A100 availability is not guaranteed by a credit balance. Model and Judge run in separate memory phases. A failed smoke run is a blocker to scaling, not a reason to spend on more candidates.

In [ ]:
from google.colab import userdata
ENV["HF_TOKEN"] = userdata.get("HF_TOKEN")
install_project("sam")
subprocess.run([UV, "run", "--no-sync", "--project", str(PROJECTS["sam"]), "python", "-c",
                "import torch; import sam_audio; assert torch.cuda.is_available(), 'Select a GPU runtime'; "
                "x = torch.ones((32, 32), device='cuda'); y = x @ x; torch.cuda.synchronize(); assert float(y[0, 0]) == 32; import logging; logging.warning('CUDA kernel passed. GPU: %s, torch: %s', torch.cuda.get_device_name(0), torch.__version__)"],
               env=ENV, check=True, timeout=120)
run_stage("sam", "sam")

23:27:01 | INFO    | sam installation and CPU import/native-library check passed


## 7. Paired Demucs baseline
CPU-only fallback, six stems, deterministic `shifts=0`. This differs from the production CLI default; the experiment records it explicitly. Do not compare its `other` output with a specific lead as if they had identical target definitions. Switch to a CPU runtime after SAM if you will spend time inspecting/transcribing; rerun workspace, configuration, environment and worker cells after reconnecting.

In [ ]:
install_project("demucs")
run_stage("demucs", "demucs")
run_stage("report", "pitch")  # Separation-only report is available before transcription.

23:33:34 | INFO    | demucs installation and CPU import/native-library check passed


## 8. Optional Basic Pitch diagnostic
Same decoder/gain policy for reference, mix, SAM candidates and the relevant Demucs stem. Activations are cached for later decoder experiments. No Demucs register bounds, majority voting, easy arrangement or application `pitch_confidence` is applied here. Results must not be presented as the production Learn score. Reference stem transcription is a diagnostic comparison, not a guaranteed ceiling.

In [ ]:
run_stage("transcribe", "pitch")
run_stage("report", "pitch")

## 9. Read reports and audition
Keep individual cases and song IDs. A two-draw smoke run is not a quality benchmark. SI-SDR alone can penalize generative phase changes; listen to target identity, leakage and artifacts. Judge is not a calibrated correctness probability. F1 is null when independent note labels are absent.

In [ ]:
import csv
from IPython.display import Audio, display
report_paths = sorted(RUN.glob("*/sam-audio-*/report.csv"))
for path in report_paths:
    with path.open() as handle:
        for row in csv.DictReader(handle):
            log.info("%s", row)
# Audio normalizes loudness for listening only; persisted measurement WAVs stay unchanged.
AUDITION_PATH = ""  # Paste an absolute candidate WAV path from RUN.
if AUDITION_PATH:
    selected = Path(AUDITION_PATH).resolve()
    if not selected.is_relative_to(RUN.resolve()):
        raise ValueError("Choose a result inside RUN")
    display(Audio(filename=str(selected)))

23:36:09 | INFO    | {'case': 'light_guitar_rhythm', 'song': '06 Let The Light In', 'split': 'dev', 'instrument': 'guitar', 'candidate': 'mix', 'si_sdr_db': '-14.219823537830926', 'si_sdri_db': '0.0', 'note_f1': '', 'pitch_class_f1': ''}
23:36:09 | INFO    | {'case': 'light_guitar_rhythm', 'song': '06 Let The Light In', 'split': 'dev', 'instrument': 'guitar', 'candidate': 'reference', 'si_sdr_db': '148.67114792189537', 'si_sdri_db': '162.8909714597263', 'note_f1': '', 'pitch_class_f1': ''}
23:36:09 | INFO    | {'case': 'light_guitar_rhythm', 'song': '06 Let The Light In', 'split': 'dev', 'instrument': 'guitar', 'candidate': 'candidate-0', 'si_sdr_db': '-26.33272199699693', 'si_sdri_db': '-12.112898459166004', 'note_f1': '', 'pitch_class_f1': ''}
23:36:09 | INFO    | {'case': 'light_guitar_rhythm', 'song': '06 Let The Light In', 'split': 'dev', 'instrument': 'guitar', 'candidate': 'candidate-1', 'si_sdr_db': '-25.549566306992308', 'si_sdri_db': '-11.329742769161381', 'note_f1': '', 'pit

## 10. Export evidence, record actual units, disconnect
Send `results.zip` back with the resource-panel units consumed, failure messages (without tokens), GPU model and your listening observations. The archive contains run configuration, pinned environments, source, candidate audio, activations and per-case reports; it excludes the full dataset and model cache.

After reviewing the smoke run, create another RUN_ID for the selected development batch. Keep held-out material untouched until a policy is frozen. Set units-after manually; no supported billing API is assumed. The final cell disconnects the runtime to release resources, so download/export first.

In [ ]:
from google.colab import files
UNITS_AFTER = None  # Read the actual current balance from Colab, e.g. 216.7.
usage = {"units_before": UNITS_BEFORE, "units_after": UNITS_AFTER,
         "units_used": None if UNITS_AFTER is None else UNITS_BEFORE - UNITS_AFTER}
(RUN / "usage.json").write_text(json.dumps(usage, indent=2))
archive_path = shutil.make_archive(str(BASE / (RUN_ID + "-results")), "zip", root_dir=RUN)
log.info("Evidence saved: %s", archive_path)
files.download(archive_path)

23:36:57 | INFO    | Evidence saved: /content/drive/MyDrive/mixray-quality/dev-003-results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# E3 repeated with a common length, then E4: is the bass default about wording or about mixtures?
E34_SOURCE = r'''"""Judge target-specificity, and a phrasing sweep with a solo-source control."""

from __future__ import annotations

import json
import math
import sys
import time
from pathlib import Path

import numpy as np
import soundfile as sf

RATE = 48000
SEED = 1000
STEMS = ("bass", "drums", "guitar", "piano", "vocals", "other")
AXES = ("overall", "precision", "recall", "faithfulness")
# The same instrument, written the way different caption styles would write it.
PHRASINGS = (
    "piano",
    "a piano",
    "piano playing",
    "the sound of a piano",
    "acoustic piano",
    "grand piano",
    "keyboard",
    "bass guitar",
)


def mono(path: Path, length: int | None = None) -> np.ndarray:
    import librosa

    audio, rate = sf.read(str(path), dtype="float64", always_2d=True)
    samples = audio.mean(axis=1)
    if rate != RATE:
        samples = librosa.resample(samples, orig_sr=rate, target_sr=RATE)
    samples = np.asarray(samples, dtype=np.float64)
    # Resampling 44.1 kHz Demucs output overshoots the 48 kHz clip, and Judge never truncates.
    if length is not None:
        if len(samples) < length:
            raise ValueError(f"{path.name}: {len(samples)} samples, expected at least {length}")
        samples = samples[:length]
    return samples


def si_sdr(reference: np.ndarray, estimate: np.ndarray) -> float:
    shared = min(len(reference), len(estimate))
    target = reference[:shared] - reference[:shared].mean()
    audio = estimate[:shared] - estimate[:shared].mean()
    power = float(target @ target)
    if power < 1e-12:
        return float("nan")
    projection = target * float(audio @ target) / power
    error = audio - projection
    return 10.0 * np.log10(
        (float(projection @ projection) + 1e-12) / (float(error @ error) + 1e-12)
    )


def snapshot_for(case_dir: Path, model_id: str) -> str:
    from huggingface_hub import snapshot_download

    recorded = case_dir / model_id.split("/")[-1] / "model.json"
    revision = json.loads(recorded.read_text())["revision"] if recorded.is_file() else None
    return snapshot_download(
        model_id, revision=revision, allow_patterns=["*.json", "*.pt", "*.safetensors"]
    )


def demucs_stems(case_dir: Path, length: int) -> dict[str, np.ndarray]:
    found = {}
    for stem in STEMS:
        path = case_dir / "demucs" / "htdemucs_6s" / "mix" / (stem + ".wav")
        if path.is_file():
            found[stem] = mono(path, length)
    return found


def table(reference: np.ndarray, stems: dict[str, np.ndarray],
          outputs: dict[str, np.ndarray], label: str) -> None:
    header = f"  {label:36s} {'vs piano':>9s} " + " ".join(f"{s:>8s}" for s in stems)
    print(header)
    print("  " + "-" * (len(header) - 2))
    for name, estimate in outputs.items():
        against = {stem: si_sdr(samples, estimate) for stem, samples in stems.items()}
        closest = max(against, key=lambda s: against[s]) if against else "-"
        row = f"  {name:36s} {si_sdr(reference, estimate):+9.2f} "
        row += " ".join(f"{against[s]:+8.2f}" for s in stems)
        print(row + f"   -> {closest}")


def judge(case_dir: Path, out_dir: Path, prompt: str) -> None:
    import torch
    from sam_audio.model.config import JudgeRankerConfig
    from sam_audio.ranking import create_ranker

    ranker = create_ranker(JudgeRankerConfig())
    if ranker is None:
        raise RuntimeError("Judge ranker unavailable")
    ranker = ranker.eval().cuda()
    mixture = mono(case_dir / "mix.wav")
    length = len(mixture)
    tensor_mix = torch.tensor(mixture, dtype=torch.float32, device="cuda").unsqueeze(0)
    subjects = [
        ("true target stem", case_dir / "reference.wav"),
        ("demucs piano", case_dir / "demucs" / "htdemucs_6s" / "mix" / "piano.wav"),
        ("demucs bass", case_dir / "demucs" / "htdemucs_6s" / "mix" / "bass.wav"),
        ("demucs vocals", case_dir / "demucs" / "htdemucs_6s" / "mix" / "vocals.wav"),
        ("the mixture", case_dir / "mix.wav"),
    ]
    subjects += [(p.stem, p) for p in sorted(out_dir.glob("*.wav"))]
    print(f"every row judged against the same request: '{prompt}'")
    header = f"  {'audio':22s} " + " ".join(f"{axis:>13s}" for axis in AXES)
    print(header)
    print("  " + "-" * (len(header) - 2))
    for name, path in subjects:
        if not path.is_file():
            continue
        tensor = torch.tensor(mono(path, length), dtype=torch.float32, device="cuda").unsqueeze(0)
        with torch.inference_mode():
            processed = ranker.processor(
                text=[prompt],
                input_audio=[tensor_mix],
                separated_audio=[tensor],
                return_tensors="pt",
                padding=True,
                sampling_rate=RATE,
            )
            judged = ranker.model(**processed.to("cuda"))
        values = {a: float(getattr(judged, a).reshape(-1)[0].cpu()) for a in AXES}
        if not all(math.isfinite(v) for v in values.values()):
            raise ValueError(name + ": Judge returned nonfinite axes")
        print(f"  {name:22s} " + " ".join(f"{values[a]:+13.3f}" for a in AXES))


def phrasings(case_dir: Path, model_id: str, out_dir: Path) -> None:
    import torch
    from sam_audio import SAMAudio, SAMAudioProcessor

    snapshot = snapshot_for(case_dir, model_id)
    model = (
        SAMAudio.from_pretrained(
            snapshot, visual_ranker=None, text_ranker=None, span_predictor=None
        )
        .eval()
        .cuda()
    )
    processor = SAMAudioProcessor.from_pretrained(snapshot)

    def draw(source: np.ndarray, prompt: str, name: str) -> np.ndarray:
        batch = processor(
            audios=[torch.tensor(source, dtype=torch.float32).unsqueeze(0)],
            descriptions=[prompt],
        ).to("cuda")
        torch.manual_seed(SEED)
        torch.cuda.manual_seed_all(SEED)
        started = time.monotonic()
        with torch.inference_mode():
            result = model.separate(batch, predict_spans=False, reranking_candidates=1)
        drawn = result.target[0].detach().float().cpu().numpy().reshape(-1)[: len(source)]
        if not np.isfinite(drawn).all():
            raise ValueError(name + ": model returned nonfinite audio")
        sf.write(str(out_dir / (name + ".wav")), drawn, RATE, subtype="FLOAT")
        print(f"  drew {name} in {time.monotonic() - started:.1f} s", flush=True)
        return drawn

    mixture = mono(case_dir / "mix.wav")
    reference = mono(case_dir / "reference.wav", len(mixture))
    stems = demucs_stems(case_dir, len(mixture))
    outputs = {}
    for index, prompt in enumerate(PHRASINGS):
        outputs[f"mix: '{prompt}'"] = draw(mixture, prompt, f"phrase-{index}")
    # Control: hand the model the clean piano alone. A working separator should return it.
    outputs["solo piano: 'piano'"] = draw(reference, "piano", "solo-piano")
    outputs["solo piano: 'the sound of a piano'"] = draw(
        reference, "the sound of a piano", "solo-verbose"
    )
    print()
    print(f"true piano vs raw mixture: {si_sdr(reference, mixture):+.2f} dB")
    table(reference, stems, outputs, "input and request")


def main() -> None:
    mode, case_dir, model_id, prompt, out_dir = (
        sys.argv[1], Path(sys.argv[2]), sys.argv[3], sys.argv[4], Path(sys.argv[5])
    )
    out_dir.mkdir(parents=True, exist_ok=True)
    if mode == "judge":
        judge(case_dir, out_dir, prompt)
    else:
        phrasings(case_dir, model_id, out_dir)


if __name__ == "__main__":
    main()
'''

E34 = WORK / "e34_probe.py"
E34.write_text(E34_SOURCE)
E34_CASE = BASE / "runs" / "dev-002" / "mag_piano_sparse"
if not (E34_CASE / "reference.wav").is_file():
    raise FileNotFoundError(f"Expected the dev-002 case audio at {E34_CASE}")


def run_e34(mode: str, out_dir: Path) -> int:
    log.info("probe: %s", mode)
    done = subprocess.run(
        [UV, "run", "--no-sync", "--project", str(PROJECTS["sam"]), "python", str(E34),
         mode, str(E34_CASE), MODEL_ID, "piano", str(out_dir)],
        env=ENV, capture_output=True, text=True, timeout=1800,
    )
    secret = ENV.get("HF_TOKEN", "")
    text = done.stdout
    if done.returncode:
        text += "\n--- stderr tail ---\n" + done.stderr[-2500:]
    if secret:
        text = text.replace(secret, "[REDACTED]")
    print(text.rstrip())
    return done.returncode


print("=" * 78)
print("E3  does Judge prefer the real piano, or any clean stem?")
print("=" * 78)
run_e34("judge", E34_CASE / "conditioning-probe")
print()
print("=" * 78)
print("E4  eight phrasings of the same request, plus a solo-piano control")
print("=" * 78)
run_e34("phrasings", E34_CASE / "prompt-probe")


E3  does Judge prefer the real piano, or any clean stem?
00:01:37 | INFO    | probe: judge
every row judged against the same request: 'piano'
  audio                        overall     precision        recall  faithfulness
  ------------------------------------------------------------------------------
  true target stem              +4.553        +4.827        +4.786        +4.206
  demucs piano                  +1.490        +2.965        +4.153        +1.947
  demucs bass                   +4.955        +4.843        +4.962        +4.103
  demucs vocals                 +1.092        +2.939        +1.105        +1.310
  the mixture                   +1.063        +1.004        +4.996        +5.045
  current                       +5.268        +5.161        +5.019        +4.234
  span_loaded                   +5.268        +5.161        +5.019        +4.234
  span_predicted                +5.236        +5.166        +4.935        +4.048

E4  eight phrasings of the same request, plus a

0

In [ ]:
# E6: how many requested concepts can the model return at all? One solo source per concept.
E6_SOURCE = r'''"""Identity sweep: hand the model one isolated source and ask for it by name."""

from __future__ import annotations

import json
import sys
import time
from pathlib import Path

import numpy as np
import soundfile as sf

RATE = 48000
SEED = 1000
TARGET_PEAK = 0.891
# Each Demucs stem paired with the wording a user would most plainly type for it.
REQUESTS = (
    ("bass", "bass guitar"),
    ("vocals", "lead vocals"),
    ("drums", "drums"),
    ("guitar", "electric guitar"),
    ("piano", "piano"),
    ("other", "synthesizer"),
)


def mono(path: Path, length: int | None = None) -> np.ndarray:
    import librosa

    audio, rate = sf.read(str(path), dtype="float64", always_2d=True)
    samples = audio.mean(axis=1)
    if rate != RATE:
        samples = librosa.resample(samples, orig_sr=rate, target_sr=RATE)
    samples = np.asarray(samples, dtype=np.float64)
    return samples if length is None else samples[:length]


def peak_normalized(samples: np.ndarray) -> np.ndarray:
    peak = float(np.max(np.abs(samples)))
    if peak < 1e-9:
        raise ValueError("Cannot normalize silence")
    return samples * (TARGET_PEAK / peak)


def si_sdr(reference: np.ndarray, estimate: np.ndarray) -> float:
    shared = min(len(reference), len(estimate))
    target = reference[:shared] - reference[:shared].mean()
    audio = estimate[:shared] - estimate[:shared].mean()
    power = float(target @ target)
    if power < 1e-12:
        return float("nan")
    projection = target * float(audio @ target) / power
    error = audio - projection
    return 10.0 * np.log10(
        (float(projection @ projection) + 1e-12) / (float(error @ error) + 1e-12)
    )


def main() -> None:
    import torch
    from huggingface_hub import snapshot_download
    from sam_audio import SAMAudio, SAMAudioProcessor

    case_dir, model_id, out_dir = Path(sys.argv[1]), sys.argv[2], Path(sys.argv[3])
    out_dir.mkdir(parents=True, exist_ok=True)
    recorded = case_dir / model_id.split("/")[-1] / "model.json"
    revision = json.loads(recorded.read_text())["revision"] if recorded.is_file() else None
    snapshot = snapshot_download(
        model_id, revision=revision, allow_patterns=["*.json", "*.pt", "*.safetensors"]
    )
    model = (
        SAMAudio.from_pretrained(
            snapshot, visual_ranker=None, text_ranker=None, span_predictor=None
        )
        .eval()
        .cuda()
    )
    processor = SAMAudioProcessor.from_pretrained(snapshot)
    length = len(mono(case_dir / "mix.wav"))

    rows = []
    for stem, prompt in REQUESTS:
        path = case_dir / "demucs" / "htdemucs_6s" / "mix" / (stem + ".wav")
        if not path.is_file():
            continue
        raw = mono(path, length)
        if float(np.max(np.abs(raw))) < 1e-6:
            print(f"  skipped {stem}: silent stem")
            continue
        source = peak_normalized(raw)
        batch = processor(
            audios=[torch.tensor(source, dtype=torch.float32).unsqueeze(0)],
            descriptions=[prompt],
        ).to("cuda")
        torch.manual_seed(SEED)
        torch.cuda.manual_seed_all(SEED)
        started = time.monotonic()
        with torch.inference_mode():
            result = model.separate(batch, predict_spans=False, reranking_candidates=1)
        drawn = result.target[0].detach().float().cpu().numpy().reshape(-1)[: len(source)]
        if not np.isfinite(drawn).all():
            raise ValueError(stem + ": model returned nonfinite audio")
        sf.write(str(out_dir / (stem + ".wav")), drawn, RATE, subtype="FLOAT")
        rows.append((stem, prompt, si_sdr(source, drawn), float(np.max(np.abs(drawn)))))
        print(f"  drew {stem} in {time.monotonic() - started:.1f} s", flush=True)

    print()
    print("each row: the model was handed ONLY that source and asked for it by name")
    header = f"  {'source':10s} {'request':20s} {'returned it':>12s} {'out peak':>9s}  verdict"
    print(header)
    print("  " + "-" * (len(header) - 2))
    for stem, prompt, score, peak in rows:
        verdict = "returned" if score > 5.0 else ("partial" if score > -5.0 else "destroyed")
        print(f"  {stem:10s} {prompt:20s} {score:+12.2f} {peak:9.5f}  {verdict}")


if __name__ == "__main__":
    main()
'''

E6 = WORK / "e6_probe.py"
E6.write_text(E6_SOURCE)
E6_CASE = BASE / "runs" / "dev-002" / "mag_piano_sparse"
done = subprocess.run(
    [UV, "run", "--no-sync", "--project", str(PROJECTS["sam"]), "python", str(E6),
     str(E6_CASE), MODEL_ID, str(E6_CASE / "identity-probe")],
    env=ENV, capture_output=True, text=True, timeout=1800,
)
secret = ENV.get("HF_TOKEN", "")
report = done.stdout
if done.returncode:
    report += "\n--- stderr tail ---\n" + done.stderr[-2500:]
if secret:
    report = report.replace(secret, "[REDACTED]")
print("=" * 78)
print("E6  which requested concepts can the model return at all?")
print("=" * 78)
print(report.rstrip())


E6  which requested concepts can the model return at all?
  drew bass in 11.1 s
  drew vocals in 10.3 s
  drew drums in 10.3 s
  drew guitar in 10.3 s
  drew piano in 10.3 s
  drew other in 10.3 s

each row: the model was handed ONLY that source and asked for it by name
  source     request               returned it  out peak  verdict
  ---------------------------------------------------------------
  bass       bass guitar                +14.61   0.56712  returned
  vocals     lead vocals                -75.35   0.20804  destroyed
  drums      drums                       +9.75   0.56885  returned
  guitar     electric guitar             +8.07   0.60062  returned
  piano      piano                      -23.96   0.45359  destroyed
  other      synthesizer                -23.19   0.59198  destroyed


In [ ]:
from google.colab import runtime
runtime.unassign()